# Recuperación — Listas Enlazadas, Pilas y Colas

**Nombre:** ____________________ **Documento:** ____________________

**Fecha:** 2 de octubre de 2026 **Hora límite de evaluación:** 3:30 p.m.

## Instrucciones

1. Lea con mucha atención todos los puntos antes de empezar a resolver.
2. Este notebook **no se envía por correo**. Todos los puntos se evalúan **durante la sesión**: cuando tenga un punto resuelto, llame al profesor para que lo revise con usted y sustente su solución.
3. Todo punto que no haya sido evaluado antes de las **3:30 p.m.** no tendrá ningún valor, aunque esté resuelto.
4. **Valor de cada punto:** cada punto empieza valiendo **1.7**. Por cada persona a la que se le evalúe correctamente ese punto, su valor baja **0.1** para las siguientes, hasta llegar a **1.0**; de ahí en adelante, quienes lo resuelvan obtienen **1.0**. Es decir, la primera persona en resolver un punto obtiene 1.7, la segunda 1.6, … y de la octava en adelante 1.0. El conteo es independiente para cada punto.
5. El uso de **typehints** en todas las funciones es obligatorio.
6. Cada punto debe incluir el **análisis de complejidad temporal y espacial** de la solución, y se sustenta en el momento de la evaluación.
7. **Estructuras permitidas:** en cada punto solo puede usar como estructuras auxiliares las implementaciones del curso que aparecen al inicio de este notebook: **colas** en los puntos 1 y 2, **pilas** en los puntos 3 y 4 y **listas enlazadas** en los puntos 5 y 6. No se permite usar `list`, `dict`, `set`, `deque`, `heapq` ni otras colecciones como estructura auxiliar; sí puede usarlas para construir la salida cuando el punto lo pida. Los casos de prueba vienen como datos de Python: construir la estructura a partir de ellos es parte de la solución.

## Reglas antifraude

- La recuperación es individual.
- No está permitido comunicarse con otras personas durante la sesión por ningún medio.
- No está permitido compartir código ni archivos con otros estudiantes.
- Soluciones idénticas o con una estructura sospechosamente similar entre estudiantes se tratarán como posible fraude.
- Cualquier intento de fraude se tramitará según el reglamento estudiantil de la universidad.

Algunos casos de prueba usan tamaños cercanos a los máximos de las restricciones.

# Clases base

## Pila y cola

In [ ]:
from typing import Any, List, Optional


class EmptyStack(Exception):
    ...


class EmptyQueue(Exception):
    ...


class Stack:
    """Pila (LIFO): push agrega al tope, pop retorna y elimina el tope."""

    def __init__(self) -> None:
        self.__stack: List[Any] = []

    def push(self, element: Any) -> None:
        self.__stack.append(element)

    def pop(self) -> Any:
        if len(self.__stack) == 0:
            raise EmptyStack("No puedes hacer pop... La pila está vacía")
        return self.__stack.pop()

    def peek(self) -> Any:
        if len(self.__stack) == 0:
            raise EmptyStack("No puedes hacer peek... La pila está vacía")
        return self.__stack[-1]

    def add_from_list(self, data: List[Any]) -> None:
        for item in data:
            self.push(item)

    def __repr__(self) -> str:
        return f"{self.__stack}"

    def __len__(self) -> int:
        return len(self.__stack)


class Queue:
    """Cola (FIFO): enqueue agrega al final, dequeue retorna y elimina el primero."""

    def __init__(self) -> None:
        self.__queue: List[Any] = []

    def enqueue(self, element: Any) -> None:
        self.__queue.append(element)

    def dequeue(self) -> Any:
        if len(self.__queue) == 0:
            raise EmptyQueue("Cola vacía... No podés hacer dequeue")
        return self.__queue.pop(0)

    def peek(self) -> Any:
        if len(self.__queue) == 0:
            raise EmptyQueue("Cola vacía... No podés hacer peek")
        return self.__queue[0]

    def __repr__(self) -> str:
        return str(self.__queue)

    def __len__(self) -> int:
        return len(self.__queue)


## Lista enlazada simple

In [ ]:
from typing import Any, Optional


class NodeSingly:
    def __init__(self, data: Any, next: Optional["NodeSingly"] = None) -> None:
        self.data = data
        self.next = next

    def __repr__(self) -> str:
        return f"{self.data}"


class SinglyLinkedList:
    def __init__(self) -> None:
        self.head: Optional[NodeSingly] = None
        self.tail: Optional[NodeSingly] = None
        self.size: int = 0

    def append(self, data: Any) -> None:
        new_node = NodeSingly(data)
        if not self.head:
            self.head = self.tail = new_node
        else:
            assert self.tail is not None
            self.tail.next = new_node
            self.tail = new_node
        self.size += 1

    def delete_and_return_first(self) -> Any:
        if self.head is None:
            return None
        old_head = self.head
        self.head = self.head.next
        old_head.next = None
        if self.head is None:
            self.tail = None
        self.size -= 1
        return old_head.data

    def delete_and_return_last(self) -> Any:
        if self.size == 0:
            return None
        if self.size == 1:
            old_tail = self.head
            self.head = None
            self.tail = None
            self.size -= 1
            return old_tail.data

        current = self.head
        while current.next != self.tail:
            current = current.next

        old_tail = self.tail
        self.tail = current
        self.tail.next = None
        self.size -= 1
        return old_tail.data

    def __repr__(self) -> str:
        values = []
        current = self.head
        while current:
            values.append(str(current.data))
            current = current.next
        return " → ".join(values) if values else "[]"

    def __len__(self) -> int:
        return self.size


---

## Punto 1 — Fila para comprar boletas

Hay `n` personas en una fila para comprar boletas de un concierto. La persona `0` está al frente de la fila y la persona `n - 1` está al final. Se conoce cuántas boletas quiere comprar cada persona: la persona `i` quiere comprar `tiquetes[i]` boletas.

Cada persona tarda exactamente **1 segundo** en comprar **una** boleta, y solo puede comprar una boleta cada vez que llega a la taquilla. Si después de comprarla todavía le faltan boletas, debe volver a hacer la fila **por el final** (instantáneamente). Si ya no le faltan boletas, se retira de la fila.

Dada la cantidad de boletas de cada persona y la posición `k` de una persona en la fila inicial, determine cuántos segundos pasan hasta que la persona `k` termina de comprar todas sus boletas.

**Entrada:** la lista `tiquetes` con la cantidad de boletas que quiere cada persona, en el orden de la fila, y el entero `k`.

**Salida:** un entero con el tiempo, en segundos, que tarda la persona `k` en terminar de comprar.

**Restricciones:**
- `1 ≤ n ≤ 100`
- `1 ≤ tiquetes[i] ≤ 100`
- `0 ≤ k < n`

**Ejemplo:** con `tiquetes = [2, 3, 2]` y `k = 2`, en la primera vuelta cada persona compra una boleta y la fila queda `[1, 2, 1]`; en la segunda vuelta cada persona compra otra y la persona `2` termina. Han pasado `6` segundos.

In [ ]:
casos_ej_001 = [
    {"entrada": {"tiquetes": [2, 3, 2], "k": 2}, "salida": 6},
    {"entrada": {"tiquetes": [5, 1, 1, 1], "k": 0}, "salida": 8},
    {"entrada": {"tiquetes": [1], "k": 0}, "salida": 1},
    {"entrada": {"tiquetes": [1, 1, 1], "k": 2}, "salida": 3},
    {"entrada": {"tiquetes": [3, 1, 4, 1, 5], "k": 2}, "salida": 12},
    {"entrada": {"tiquetes": [2, 6, 3, 4, 5], "k": 2}, "salida": 12},
    {"entrada": {"tiquetes": [4, 4, 4], "k": 1}, "salida": 11},
    # caso grande
    {"entrada": {"tiquetes": [100] * 100, "k": 99}, "salida": 10000},
    # caso grande
    {"entrada": {"tiquetes": [100] * 100, "k": 0}, "salida": 9901},
    # caso grande
    {"entrada": {"tiquetes": [1] + [100] * 99, "k": 0}, "salida": 1},
    # caso grande
    {"entrada": {"tiquetes": [100] * 99 + [1], "k": 99}, "salida": 100},
]

**Análisis de complejidad (temporal y espacial):**


---

## Punto 2 — Emparejamiento de viajes compartidos

Una plataforma de viajes compartidos debe emparejar conductores con pasajeros **según el orden de llegada**. Diseñe un sistema que, empezando vacío, soporte las siguientes operaciones:

- `agregar_pasajero(id)`: registra a un nuevo pasajero que queda esperando un conductor.
- `agregar_conductor(id)`: registra a un nuevo conductor que queda disponible.
- `emparejar()`: empareja al conductor que lleva **más tiempo disponible** con el pasajero que lleva **más tiempo esperando**; ambos salen del sistema. El resultado es el par `[id_conductor, id_pasajero]`. Si no hay al menos un conductor disponible y al menos un pasajero esperando, no se empareja a nadie y el resultado es `[-1, -1]`.
- `cancelar_pasajero(id)`: si el pasajero con ese identificador está esperando (fue registrado y todavía no ha sido emparejado), cancela su solicitud y sale del sistema. En cualquier otro caso no ocurre nada.

**Entrada:** una secuencia de operaciones; cada operación es una tupla con el nombre de la operación y, cuando aplica, el identificador.

**Salida:** una lista con el resultado de cada operación, en el mismo orden. Las operaciones `emparejar` producen el par descrito; las demás producen `None`.

**Restricciones:**
- `1 ≤ id ≤ 1000`
- Cada identificador de pasajero es único entre los pasajeros y se agrega a lo sumo una vez; lo mismo ocurre con los conductores. Un pasajero y un conductor sí pueden tener el mismo identificador.
- Se realizan a lo sumo `1000` operaciones.

In [ ]:
casos_ej_002 = [
    {
        "entrada": [("agregar_pasajero", 3), ("agregar_conductor", 2), ("agregar_pasajero", 1),
                    ("emparejar",), ("agregar_conductor", 5), ("cancelar_pasajero", 3),
                    ("emparejar",), ("emparejar",)],
        "salida": [None, None, None, [2, 3], None, None, [5, 1], [-1, -1]],
    },
    {
        "entrada": [("agregar_pasajero", 8), ("agregar_conductor", 8), ("agregar_conductor", 6),
                    ("emparejar",), ("agregar_pasajero", 2), ("cancelar_pasajero", 2), ("emparejar",)],
        "salida": [None, None, None, [8, 8], None, None, [-1, -1]],
    },
    {
        "entrada": [("emparejar",)],
        "salida": [[-1, -1]],
    },
    {
        "entrada": [("agregar_conductor", 1), ("agregar_conductor", 2), ("emparejar",)],
        "salida": [None, None, [-1, -1]],
    },
    {
        "entrada": [("agregar_pasajero", 4), ("agregar_pasajero", 7), ("agregar_pasajero", 9),
                    ("cancelar_pasajero", 7), ("agregar_conductor", 1), ("agregar_conductor", 2),
                    ("emparejar",), ("emparejar",), ("emparejar",)],
        "salida": [None, None, None, None, None, None, [1, 4], [2, 9], [-1, -1]],
    },
    {
        "entrada": [("agregar_pasajero", 5), ("agregar_conductor", 10), ("emparejar",),
                    ("cancelar_pasajero", 5), ("cancelar_pasajero", 42), ("agregar_pasajero", 6),
                    ("agregar_conductor", 11), ("emparejar",)],
        "salida": [None, None, [10, 5], None, None, None, None, [11, 6]],
    },
    {
        "entrada": [("agregar_pasajero", 1), ("agregar_pasajero", 2), ("cancelar_pasajero", 1),
                    ("cancelar_pasajero", 2), ("agregar_conductor", 3), ("emparejar",)],
        "salida": [None, None, None, None, None, [-1, -1]],
    },
    # caso grande
    {
        "entrada": [("agregar_pasajero", i) for i in range(1, 301)]
                   + [("cancelar_pasajero", i) for i in range(2, 301, 2)]
                   + [("agregar_conductor", i) for i in range(1, 201)]
                   + [("emparejar",)] * 350,
        "salida": [None] * 650
                  + [[i, 2 * i - 1] for i in range(1, 151)]
                  + [[-1, -1]] * 200,
    },
    # caso grande
    {
        "entrada": [("agregar_conductor", i) for i in range(1000, 500, -1)]
                   + [("agregar_pasajero", i) for i in range(1, 500)]
                   + [("emparejar",)],
        "salida": [None] * 999 + [[1000, 1]],
    },
]

**Análisis de complejidad (temporal y espacial):**


---

## Punto 3 — El número más pequeño según un patrón

Se da una cadena `patron` de longitud `n` formada únicamente por los caracteres `'I'` (*increasing*, aumenta) y `'D'` (*decreasing*, disminuye).

Se quiere construir una cadena `num` de longitud `n + 1` que cumpla:

- `num` está formada por los dígitos del `'1'` al `'9'`, y cada dígito se usa **a lo sumo una vez**.
- Si `patron[i] == 'I'`, entonces `num[i] < num[i + 1]`.
- Si `patron[i] == 'D'`, entonces `num[i] > num[i + 1]`.

Entre todas las cadenas que cumplen estas condiciones, determine la **lexicográficamente menor**.

**Entrada:** la cadena `patron`.

**Salida:** la cadena `num` lexicográficamente menor que cumple las condiciones.

**Restricciones:**
- `1 ≤ n ≤ 8`
- `patron` solo contiene los caracteres `'I'` y `'D'`.

**Ejemplo:** para `patron = "IIIDIDDD"`, cadenas como `"245639871"`, `"135749862"` y `"123849765"` cumplen las condiciones, pero la menor de todas es `"123549876"`.

In [ ]:
casos_ej_003 = [
    {"entrada": "IIIDIDDD", "salida": "123549876"},
    {"entrada": "DDD", "salida": "4321"},
    {"entrada": "I", "salida": "12"},
    {"entrada": "D", "salida": "21"},
    {"entrada": "IDID", "salida": "13254"},
    {"entrada": "IIDDD", "salida": "126543"},
    {"entrada": "DDIIDD", "salida": "3214765"},
    # caso grande
    {"entrada": "DDDDDDDD", "salida": "987654321"},
    # caso grande
    {"entrada": "IIIIIIII", "salida": "123456789"},
    # caso grande
    {"entrada": "DIDIDIDI", "salida": "214365879"},
]

**Análisis de complejidad (temporal y espacial):**


---

## Punto 4 — Invertir el comienzo de una palabra

Dada una cadena `palabra` y un carácter `ch`, invierta el segmento de `palabra` que va desde el índice `0` hasta el índice de la **primera aparición** de `ch`, incluyéndolo. Si `ch` no aparece en `palabra`, la palabra no cambia.

**Entrada:** la cadena `palabra` y el carácter `ch`.

**Salida:** la cadena resultante.

**Restricciones:**
- `1 ≤ longitud de palabra ≤ 250`
- `palabra` está formada por letras minúsculas del alfabeto inglés.
- `ch` es una letra minúscula del alfabeto inglés.

**Ejemplo:** con `palabra = "abcdefd"` y `ch = "d"`, la primera aparición de `"d"` está en el índice `3`; al invertir el segmento `"abcd"` se obtiene `"dcbaefd"`.

In [ ]:
casos_ej_004 = [
    {"entrada": {"palabra": "abcdefd", "ch": "d"}, "salida": "dcbaefd"},
    {"entrada": {"palabra": "xyxzxe", "ch": "z"}, "salida": "zxyxxe"},
    {"entrada": {"palabra": "abcd", "ch": "z"}, "salida": "abcd"},
    {"entrada": {"palabra": "a", "ch": "a"}, "salida": "a"},
    {"entrada": {"palabra": "aab", "ch": "a"}, "salida": "aab"},
    {"entrada": {"palabra": "racecar", "ch": "e"}, "salida": "ecarcar"},
    {"entrada": {"palabra": "abcba", "ch": "a"}, "salida": "abcba"},
    # caso grande
    {"entrada": {"palabra": "a" * 249 + "b", "ch": "b"}, "salida": "b" + "a" * 249},
    # caso grande
    {"entrada": {"palabra": "abcdefghij" * 25, "ch": "j"}, "salida": "jihgfedcba" + "abcdefghij" * 24},
    # caso grande
    {"entrada": {"palabra": "m" * 250, "ch": "q"}, "salida": "m" * 250},
]

**Análisis de complejidad (temporal y espacial):**


---

## Punto 5 — Purga de descensos

Se tiene una secuencia de enteros almacenada en una **lista enlazada simple**. En cada **paso** se eliminan, **simultáneamente**, todos los nodos cuyo valor es **estrictamente menor** que el valor del nodo inmediatamente anterior en la lista (el primer nodo nunca se elimina, porque no tiene anterior). Las comparaciones de un paso se hacen todas sobre la lista tal como estaba al comenzar ese paso.

Determine cuántos pasos se realizan hasta que la lista quede ordenada de forma no decreciente, es decir, hasta que un paso ya no elimine ningún nodo (ese último paso no se cuenta).

**Entrada:** la secuencia de enteros, en el orden de la lista.

**Salida:** un entero con la cantidad de pasos.

**Restricciones:**
- `1 ≤ n ≤ 10^5`, donde `n` es la cantidad de nodos.
- `1 ≤ valor ≤ 10^9`

**Ejemplo:** con `5 → 3 → 4 → 4 → 7 → 3 → 6 → 11 → 8 → 5 → 11`:
- Paso 1: se eliminan `3`, `3`, `8` y `5`, y queda `5 → 4 → 4 → 7 → 6 → 11 → 11`.
- Paso 2: se eliminan `4` y `6`, y queda `5 → 4 → 7 → 11 → 11`.
- Paso 3: se elimina `4`, y queda `5 → 7 → 11 → 11`, que ya es no decreciente.

La respuesta es `3`.

In [ ]:
casos_ej_005 = [
    {"entrada": [5, 3, 4, 4, 7, 3, 6, 11, 8, 5, 11], "salida": 3},
    {"entrada": [4, 5, 7, 7, 13], "salida": 0},
    {"entrada": [1], "salida": 0},
    {"entrada": [7, 7, 7], "salida": 0},
    {"entrada": [3, 2, 1], "salida": 1},
    {"entrada": [10, 1, 2, 3, 4, 5, 6, 7, 8, 9], "salida": 9},
    {"entrada": [10, 1, 9, 2, 8, 3], "salida": 2},
    {"entrada": [5, 1, 2, 6, 1, 2, 3], "salida": 3},
    # caso grande
    {"entrada": [10**5] + list(range(1, 10**5)), "salida": 99999},
    # caso grande
    {"entrada": list(range(10**5, 0, -1)), "salida": 1},
    # caso grande
    {"entrada": [10**9] + list(range(1, 50_001)) + [10**9 - 1] + list(range(1, 49_999)), "salida": 50001},
]

**Análisis de complejidad (temporal y espacial):**


---

## Punto 6 — Repartir una lista en partes

Dada una **lista enlazada simple** y un entero `k`, divida la lista en `k` partes **consecutivas**.

- Las longitudes de las partes deben ser lo más parecidas posible: dos partes cualesquiera no pueden diferir en más de un nodo. Algunas partes pueden quedar vacías.
- Las partes deben respetar el orden de la lista original, y una parte que aparece antes debe tener una longitud **mayor o igual** que cualquier parte que aparezca después.
- Las partes se forman con los nodos de la lista original, reorganizando sus enlaces, sin crear nodos nuevos.

**Entrada:** la secuencia de valores de la lista, en orden, y el entero `k`.

**Salida:** las `k` partes, en orden. En los casos de prueba cada parte se muestra como la secuencia de sus valores, y una parte vacía como `[]`.

**Restricciones:**
- `0 ≤ n ≤ 1000`, donde `n` es la cantidad de nodos.
- `0 ≤ valor ≤ 1000`
- `1 ≤ k ≤ 50`

**Ejemplo:** la lista `1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10` con `k = 3` se divide en `1 → 2 → 3 → 4`, `5 → 6 → 7` y `8 → 9 → 10`.

In [ ]:
casos_ej_006 = [
    {"entrada": {"lista": [1, 2, 3], "k": 5}, "salida": [[1], [2], [3], [], []]},
    {"entrada": {"lista": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10], "k": 3}, "salida": [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]},
    {"entrada": {"lista": [], "k": 3}, "salida": [[], [], []]},
    {"entrada": {"lista": [7], "k": 1}, "salida": [[7]]},
    {"entrada": {"lista": [4, 0, 4, 0], "k": 1}, "salida": [[4, 0, 4, 0]]},
    {"entrada": {"lista": [9, 8, 7, 6, 5, 4, 3], "k": 2}, "salida": [[9, 8, 7, 6], [5, 4, 3]]},
    {"entrada": {"lista": [1, 2, 3, 4, 5, 6], "k": 4}, "salida": [[1, 2], [3, 4], [5], [6]]},
    # caso grande
    {"entrada": {"lista": list(range(1000)), "k": 50},
     "salida": [list(range(20 * i, 20 * i + 20)) for i in range(50)]},
    # caso grande
    {"entrada": {"lista": list(range(1000)), "k": 7},
     "salida": [list(range(0, 143)), list(range(143, 286)), list(range(286, 429)), list(range(429, 572)),
                list(range(572, 715)), list(range(715, 858)), list(range(858, 1000))]},
    # caso grande
    {"entrada": {"lista": [1000] * 3, "k": 50}, "salida": [[1000], [1000], [1000]] + [[]] * 47},
]

**Análisis de complejidad (temporal y espacial):**
